In [ ]:
# Метрика оценка поиска - преобразовываем списки в метрики => 16

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

for doc in documents:
  if doc["course"] == "llm-zoomcamp":
    documents_llm.append(doc)

documents = documents_llm
index = build_index(documents) 

In [4]:
def text_search(query):
  boost_dict = {"question": 3.0, "section": 0.5}

  return index.search(
    query,
    num_results = 5,
    boost_dict = boost_dict
  )

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-data.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [6]:
df_ground_truth.head()

,question,document
0,I found this course late — can I still enroll ...,74eb249bbf
1,"Is it too late to join the course now, or can ...",74eb249bbf
2,Can I still participate in the course even if ...,74eb249bbf
3,"If I join late, do I still have a chance to ge...",74eb249bbf
4,What’s the deadline if I want a certificate af...,74eb249bbf


In [7]:
q = ground_truth[0]
q

'''
{
  'question': 'I found this course late — can I still enroll and follow along?',
  'document': '74eb249bbf'
}
'''

"\n{\n  'question': 'I found this course late — can I still enroll and follow along?',\n  'document': '74eb249bbf'\n}\n"

In [8]:
doc_id = q["document"] 
doc_query = q["question"] 

results = text_search(query = doc_query)

In [9]:
for d in results:
  print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

04919992b3 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
69d122f12e == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False


In [10]:
relevance = []

for d in results:
  relevance.append(int(d["id"] == doc_id))

relevance

[0, 1, 0, 0, 0]

In [11]:
def compute_relevance_text(q):
  doc_id = q["document"]

  results = text_search(query = doc_query)

  relevance = []
  for d in results:
    relevance.append(int(d["id"] == doc_id))

  return relevance

q = ground_truth[0]
print(doc_query)
compute_relevance_text(q)

I found this course late — can I still enroll and follow along?


[0, 1, 0, 0, 0]

In [12]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
  relevance_total = []

  for q in tqdm(ground_truth):
    relevance = compute_relevance_text(q)
    relevance_total.append(relevance)

  return relevance_total

ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [13]:
def compute_relevance(q, search_function):
  doc_id = q["document"]
  results = search_function(query = doc_query)

  relevance = []
  for d in results:
    relevance.append(int(d["id"] == doc_id))

  return relevance


In [14]:
def compute_relevance_total(ground_truth, search_function):
  relevance_total = []

  for q in tqdm(ground_truth):
    relevance = compute_relevance(q, search_function)
    relevance_total.append(relevance)

  return relevance_total

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [15]:
relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

  0%|          | 0/765 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0,

In [16]:
print("Начало метрики оценки поиска")

Начало метрики оценки поиска


In [ ]:
# Hit Rate - показатель попаданий - в скольких случаях мы можем получить 
# документ

In [ ]:
# Каждая строка представляет собой один запрос. Если строка содержит 
# нули 1, значит, поиск нашел правильный документ где-то среди 5 лучших 
# результатов. Если строка содержит только 0, то поиск не нашел 
# нужный документ.

example = [
  [1, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [0, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [0, 0, 1, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
]

In [ ]:
# Теперь пройдемся по каждой строке и проверяем есть ли 1 или нет

cnt = 0

for line in example:
  if 1 in line:
    cnt = cnt + 1

cnt                           # 14

# Берем 14 и делим на общее кол-во примеров на которое у нас есть
cnt / len(example)            # 0.933 

0.9333333333333333

In [ ]:
# Теперь поместить всю логику в отдельную функцию
def hit_rate(relevance):
  cnt = 0

  for line in relevance:
    if 1 in line:
      cnt = cnt + 1

  return cnt / len(relevance)

hit_rate(example)          # 0.9333333333333333

0.9333333333333333

In [ ]:
# Показатель успешности поиска (Hit Rate) показывает, нашли ли мы нужный документ, 
# но не указывает, где он находится.

In [ ]:
# Mean Reciprocal Rank (MRR) - Среднеобратный ранг. 
# Он также учитывает эту позицию. Для каждого запроса 
# оценка основывается на ранге первого правильного документа:
# позиция 1: счет 1,0
# позиция 2: счет 0,5
# позиция 3: счет 0,333
# не найдено: оценка равна 0
# В приведенном примере большинство результатов поиска находятся на первой позиции. 
# Некоторые результаты поиска расположены ниже в списке.


In [ ]:
# Чтобы было понятнее
example = [
  [1, 0, 0, 0, 0],      # 100 %
  [0, 1, 0, 0, 0],      # 50 % 
  [0, 0, 1, 0, 0],      # 33 %
  [0, 0, 0, 1, 0],      # 25 %
  [0, 0, 0, 0, 1],      # 20 % 
]

#? Формула: 1 / (rank + 1)

In [25]:
sample = [
  [1, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [0, 0, 0, 0, 0],
  [0, 1, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [0, 0, 1, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
  [1, 0, 0, 0, 0],
]


In [28]:
total_score = 0.0

for line in sample:
  for rank in range(len(line)):
    if line[rank] == 1:
      score = 1 / (rank + 1)
      total_score = total_score + score
      break

total_score / len(sample)

0.8222222222222222

In [ ]:
def mrr(relevance):
  total_score = 0.0

  # line - это 1 массив
  for line in relevance:
    for rank in range(len(line)):
      if line[rank] == 1:
        formula = 1 / (rank + 1)
        total_score = total_score + formula
        break

  return total_score / len(relevance)

mrr(sample)     # 0.822

1.0
0.5
1.0
0.5
1.0
1.0
1.0
1.0
0.3333333333333333
1.0
1.0
1.0
1.0
1.0


0.8222222222222222

In [ ]:
# Теперь собираем все вместе 
def evaluate(ground_truth, search_function):
  # Получаем массив успешных документов - [0, 0, 0, 1, 0]
  relevance_total = compute_relevance_total(ground_truth, search_function)

  return {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total),
  }

evaluate(ground_truth, text_search)
# {"hit_rate": 0.899, "mrr": 0.769}